<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
Learning Loop: Agenten systematisch verbessern
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Der Learning Loop vergleicht Antworten des PMO-Assistenten auf Entwicklungsfällen, untersucht wiederkehrende Fehler und prüft eine gezielte Änderung der Anweisung. Bekannte Regressionsfälle dienen als zusätzliche Vergleichsbasis; neue zurückgehaltene Fälle bleiben bis zur abschließenden Prüfung unbenutzt. Die Ergebnisse unterstützen eine begründete Entscheidung über die Übernahme.

Das Notebook verwendet denselben PMO-Korpus und dieselbe Retrieval-Bibliothek wie der RAG-Agent und die PMO-Referenzimplementierung. Es baut einen bewusst kleinen PMO-Graphen selbstständig auf; es importiert nicht den vollständigen Referenzgraphen der Referenzimplementierung. Die Modellaufrufe und die Suche sind echt. Der Learning Loop ändert Anweisungen, nicht die Modellgewichte.

**Praxis-Transfer:** Mara Vogt beobachtet, dass der PMO-Assistent eine Projektfrage unvollständig oder mit einer unbelegten Aussage beantwortet. Vor der fachlichen Abnahme einer verbesserten Fassung benötigt sie einen Vergleich mit den bekannten Prüffällen und zusätzlichen, bisher unbenutzten Fällen. Der Bericht soll Verbesserungen und neu entstandene Fehler getrennt ausweisen.

In [ ]:
#@title 🛠️ Umgebung einrichten { display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul
!uv pip install --system -q "markitdown[all]"

import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGSMITH_PROJECT"] = "m27_learning_loop"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    copy_from_github,
    get_ipinfo,
    mermaid,
    mprint,
    setup_api_keys,
)
from genai_lib.model_config import (
    BASELINE,
    ROUTER,
    WORKER,
    CODING,
    JUDGE,
    PLANNER,
    WORKER_PREMIUM,
    FRONTIER,
    IMAGE_GENERATION,
    TRANSCRIPTION,
    TRANSCRIPTION_SEGMENTS,
    EMBEDDINGS,
)

setup_api_keys(["OPENAI_API_KEY"], create_globals=False)
check_environment()
get_ipinfo()

# LangSmith Tracing
run_cfg = {
    "tags": ["learning_loop", "pmo_briefing"],
    "metadata": {"notebook": "learning_loop_agentenverbesserung", "version": "1.0"},
}


In [ ]:
#@title 📂 Google Drive mounten (PMO-RAG) { display-mode: "form" }
from google.colab import drive

drive.mount("/content/drive")

# 1 | Übersicht
---

Ein einzelner guter Lauf zeigt noch nicht, ob ein Agent zuverlässig arbeitet. Ein Learning Loop sammelt Ergebnisse mehrerer Fälle, untersucht die Fehler und prüft eine gezielte Änderung gegen die bisherige Version.

**Lernziele:** Du kannst den PMO-Assistenten auf Entwicklungsfällen evaluieren, eine begründete Änderung seiner Anweisung erproben und sie auf zurückgehaltenen Fällen prüfen. Du unterscheidest automatische Hinweise von einer fachlichen Freigabe.

**Voraussetzungen:** State und Nodes aus den StateGraph-Grundlagen, PMO-Retrieval aus dem RAG-Agenten sowie Evaluation aus den LangSmith-Evaluations oder aus Agent Evaluation und Testing. Das Reflection-Muster überarbeitet ein einzelnes Ergebnis innerhalb eines Laufs; der Learning Loop verbessert die Anweisung für spätere Läufe.

In [ ]:
#@markdown <p><font size="4" color='green'>🔄 Learning Loop im PMO-Assistenten</font></p>
mermaid("""%%{init: {'theme': 'base', 'themeVariables': {'primaryColor': '#e0f2fe', 'primaryTextColor': '#0c4a6e', 'primaryBorderColor': '#0284c7', 'lineColor': '#475569', 'fontFamily': 'Arial'}}}%%
flowchart TD
    A([PMO-Fälle]) --> B[Baseline ausführen]
    B --> C[Ergebnisse prüfen]
    C --> D[Fehler analysieren]
    D --> E[Anweisung ändern]
    E --> F[Variante vergleichen]
    F --> G{Entwicklungsfälle bestanden}
    G -->|Ja| H[Zurückgehaltene Fälle prüfen]
    G -->|Nein| I([Änderung verwerfen])
    H --> J{Keine Regression}
    J -->|Ja| K([Änderung übernehmen])
    J -->|Nein| I
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef metric fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;
    classDef stop fill:#fee2e2,stroke:#dc2626,stroke-width:2px,color:#7f1d1d;
    class A input;
    class G,J metric;
    class K output;
    class I stop;
""")


# 2 | PMO-Korpus und Retrieval vorbereiten
---

Wir verwenden den gleichen Korpuspfad und die persistente Chroma-Collection wie die PMO-Referenzimplementierung. Der Korpus liegt in Google Drive. Falls er bereits indexiert wurde, verwendet die Bibliothek den vorhandenen Index; andernfalls indexiert sie die Dokumente.

Die Retrieval-Treffer bilden den gemeinsamen Kontext für Baseline und Variante. So kann ein Unterschied in den Antworten der geänderten Anweisung zugeordnet werden, ohne dass sich zusätzlich die gefundenen Dokumente zwischen den Varianten ändern. Die Retrieval-Qualität selbst prüfen wir separat.

In [ ]:
from pathlib import Path
from genai_lib.pmo_rag import get_pmo_vectorstore, index_pmo_corpus, format_docs

drive_root = Path("/content/drive/MyDrive/Agenten")
corpus_path = drive_root / "02_daten/01_text/korpus_pmo_assistent"
db_path = drive_root / "02_daten/05_sonstiges/chroma_pmo"
corpus_path.mkdir(parents=True, exist_ok=True)
db_path.mkdir(parents=True, exist_ok=True)

copy_from_github(
    source="ralf-42/Agenten/02_daten/01_text/korpus_pmo_assistent",
    target=str(corpus_path),
    mask="*",
)
vectorstore = get_pmo_vectorstore(persist_directory=str(db_path))
indexed_chunks = index_pmo_corpus(vectorstore, str(corpus_path))
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"Neu indexierte Chunks: {indexed_chunks}")


# 3 | Entwicklungsfälle und zurückgehaltene Fälle
---

Die Fälle orientieren sich an den PMO-Fragen und Quelldateien aus Agent Evaluation und Testing. Die ersten Fälle dienen der Analyse und Auswahl einer Änderung. Die zurückgehaltenen Fälle werden erst geprüft, nachdem die Änderung festgelegt ist. In der Praxis werden sie getrennt verwaltet, damit sie die Formulierung der Änderung nicht beeinflussen.

Jeder Fall mit Korpusantwort nennt neben der erwarteten Quelle einen erwarteten Beleg (`expected_facts`) — eine Angabe, die in einer brauchbaren Antwort vorkommen muss. Die erwarteten Quellen sind Anhaltspunkte für den Retrieval-Check. Ein Treffer kann trotzdem fachlich ungeeignet sein; eine Antwort kann auch aus einer anderen gültigen Quelle korrekt belegt sein. Deshalb werden Treffer und Antworten zusätzlich von einem Menschen beurteilt.

In [ ]:
dev_cases = [
    {"id": "dev_1", "query": "Welche Entscheidung gilt für den Prototyp?", "expected_source": "entscheidungsprotokoll_gesamt.pdf", "expected_facts": ["ChromaDB"], "out_of_corpus": False},
    {"id": "dev_2", "query": "Welche Entscheidung gilt für den produktiven Betrieb?", "expected_source": "entscheidungsprotokoll_gesamt.pdf", "expected_facts": ["D2"], "out_of_corpus": False},
    {"id": "dev_3", "query": "Welchen Status hat Risiko R2?", "expected_source": "protokoll_steuerkreis_2026-04-28.pdf", "expected_facts": ["mitigiert"], "out_of_corpus": False},
    {"id": "dev_4", "query": "Wie hoch ist das Reisebudget für das nächste Firmenfest?", "expected_source": "", "expected_facts": [], "out_of_corpus": True},
    {"id": "dev_5", "query": "Auf welche frühere Entscheidung zum Prototyp bezieht sich der Change Request CR-02?", "expected_source": "entscheidungsprotokoll_gesamt.pdf", "expected_facts": ["17.03.2026", "17. März 2026"], "out_of_corpus": False},
]
holdout_cases = [
    {"id": "test_1", "query": "Wann wurde D2 im Steuerkreis bestätigt?", "expected_source": "protokoll_steuerkreis_2026-04-14.pdf", "expected_facts": ["14.04.2026", "14. April 2026"], "out_of_corpus": False},
    {"id": "test_2", "query": "Wer gewinnt die nächste Fußballweltmeisterschaft?", "expected_source": "", "expected_facts": [], "out_of_corpus": True},
]
assert not ({c["id"] for c in dev_cases} & {c["id"] for c in holdout_cases})
assert all(case["expected_facts"] for case in dev_cases + holdout_cases if not case["out_of_corpus"]), \
    "Jeder Fall mit erwarteter Korpusantwort braucht mindestens einen Beleg in expected_facts."
print(f"Entwicklungsfälle: {len(dev_cases)}; zurückgehalten: {len(holdout_cases)}")

# 4 | PMO-Agent und Bewertung
---

Der LangGraph erhält die Frage, den bereits abgerufenen Kontext und eine Anweisung. Sein Antwort-Node ruft das konfigurierte Modell auf. Anschließend erfasst ein Prüf-Node einfache Hinweise: Wurde die erwartete Quelle gefunden und genannt? Wird bei einer Frage außerhalb des Korpus die Grenze benannt?

Die Suche wird wie in den Kursstandards ausdrücklich vor dem Graph-Aufruf ausgeführt, dann werden die Dokumente formatiert und an den Agenten übergeben. Semantische Ähnlichkeit liefert auch zu sachfremden Fragen manchmal Treffer. Ob diese Treffer die Antwort tatsächlich stützen, entscheidet nicht der Retrieval-Score, sondern muss anhand der Ausschnitte und Antworten überprüft werden.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

llm = init_chat_model(WORKER)

class LearningState(TypedDict, total=False):
    case: dict
    instruction: str
    context: str
    sources: list[str]
    answer: str
    evaluation: dict



In [ ]:
def answer_node(state: LearningState) -> dict:
    if not state["context"].strip():
        return {"answer": "Nicht im Korpus."}
    result = llm.invoke([
        SystemMessage(content=state["instruction"]),
        HumanMessage(content=f"Frage: {state['case']['query']}\n\nKorpusauszüge:\n{state['context']}"),
    ], config={**run_cfg, "run_name": "m27_pmo_antwort"})
    return {"answer": result.text}

def evaluation_node(state: LearningState) -> dict:
    case, answer = state["case"], state["answer"]
    sources = state["sources"]
    antwort, kontext = answer.casefold(), state["context"].casefold()
    if case["out_of_corpus"]:
        retrieval, source_cited, fact_in_context, fact_check = None, None, None, None
        check = "nicht im korpus" in antwort
    else:
        expected = case["expected_source"].casefold()
        retrieval = any(expected in source.casefold() for source in sources)
        source_cited = expected in antwort
        fact_in_context = any(fact.casefold() in kontext for fact in case["expected_facts"])
        fact_check = any(fact.casefold() in antwort for fact in case["expected_facts"])
        check = source_cited and fact_check
    return {"evaluation": {"id": case["id"], "retrieval_hit": retrieval, "source_cited": source_cited,
                           "fact_in_context": fact_in_context, "fact_check": fact_check,
                           "answer_check": check, "answer": answer, "sources": sources}}

In [ ]:
# Graph definieren
agent_builder = StateGraph(LearningState)

agent_builder.add_node("answer", answer_node)
agent_builder.add_node("evaluation", evaluation_node)

agent_builder.add_edge(START, "answer")
agent_builder.add_edge("answer", "evaluation")
agent_builder.add_edge("evaluation", END)

agent = agent_builder.compile()

In [ ]:
def prepare_cases(cases: list[dict]) -> list[dict]:
    prepared = []
    for case in cases:
        docs = retriever.invoke(case["query"])
        context = format_docs(docs)
        sources = [doc.metadata.get("source", "").replace("\\", "/").rsplit("/", 1)[-1] for doc in docs]
        prepared.append({"case": case, "context": context, "sources": sources})
    return prepared

def run_cases(prepared: list[dict], instruction: str) -> list[dict]:
    return [agent.invoke({**item, "instruction": instruction})["evaluation"] for item in prepared]

def zeige(wert) -> str:
    return "entfällt" if wert is None else str(wert)

def diagnose(result: dict) -> str:
    """Ordnet einen Fehlschlag der Stufe zu, an der er entstanden ist."""
    if result["answer_check"] and result["retrieval_hit"] is not False:
        return "Fall bestanden."
    if result["retrieval_hit"] is None:
        return "Fall ohne erwartete Korpusantwort → die Antwort benennt die Korpusgrenze nicht und behauptet Unbelegtes."
    if result["answer_check"]:
        return "Antwort-Check bestanden; erwartete Quelle nicht unter den Treffern → Quellenerwartung des Falls prüfen."
    if result["fact_in_context"] is False:
        return "Beleg fehlt im Kontext → Retrieval-Fehler; eine andere Anweisung ersetzt die Evidenz nicht."
    if result["fact_check"] is False:
        return "Beleg lag im Kontext, fehlt aber in der Antwort → Antwortfehler; eine Anweisungsänderung kann helfen."
    return ("Beleg genannt, Quelle fehlt → die Antwort belegt ihre Aussage nicht; "
            "eine Anweisungsänderung kann helfen.")

def show_results(results: list[dict]) -> None:
    for result in results:
        # Statuszeile und Quellen bleiben Klartext; nur die Modellantwort enthaelt Markdown
        print(f"\n{result['id']} | Antwort-Check: {result['answer_check']}"
              f" | Retrieval-Treffer: {zeige(result['retrieval_hit'])}"
              f" | Beleg im Kontext: {zeige(result['fact_in_context'])}"
              f" | Erwartete Quelle genannt: {zeige(result['source_cited'])}"
              f" | Beleg genannt: {zeige(result['fact_check'])}")
        print("Gefundene Quellen:", ", ".join(result["sources"]) or "keine")
        mprint(f"**Antwort:** {result['answer']}")
    print(f"\nAntwort-Checks bestanden: {sum(r['answer_check'] for r in results)}/{len(results)}")

Der Check prüft getrennt, ob die erwartete Quelle in der Antwort genannt wird und ob ein erwarteter Beleg aus `expected_facts` darin vorkommt. Bestanden ist ein Fall nur, wenn beides zutrifft. Zusätzlich meldet `Beleg im Kontext`, ob der erwartete Beleg überhaupt in den abgerufenen Auszügen stand. Daraus ergibt sich die Einordnung eines Fehlschlags: Fehlt der Beleg schon im Kontext, liegt ein Retrieval-Fehler vor, den keine Anweisung heilt. Stand er im Kontext und fehlt nur in der Antwort, kann eine andere Anweisung helfen. Dasselbe gilt für eine Antwort, die den Beleg nennt, aber die Quelle verschweigt.

Beide Prüfungen bleiben Textvergleiche. Sie bestätigen eine Zeichenkette, keine Aussage: „ChromaDB ist überholt“ besteht ebenso wie eine fachlich richtige Antwort. `Beleg im Kontext` hat eine eigene Grenze — die Auszüge enthalten Kopf- und Metadatenzeilen, sodass ein Beleg, der im Dateinamen oder in den Korpus-Metadaten steht, dort immer erscheint. Ein `False` beim Antwort-Check kann an einer anderen berechtigten Quelle oder an einer abweichenden Formulierung desselben Sachverhalts liegen. Für Out-of-Corpus-Fälle prüft der Test nur die Formulierung „Nicht im Korpus“; unbelegte Zusatzbehauptungen müssen manuell erkannt werden. Jedes Ergebnis braucht deshalb den Blick in Kontext und Antwort.

# 5 | Baseline ausführen und Fehler untersuchen
---

Die Baseline ist der aktuelle Stand der Anweisung: Sie bindet die Antwort an die Auszüge und verlangt bei fehlender Evidenz die Formulierung „Nicht im Korpus“. Eine Pflicht, jede Aussage einer Datei zuzuordnen, enthält sie nicht. Ob sich das in den Ergebnissen niederschlägt, zeigt erst die Analyse — ein Modell kann Quellen auch unaufgefordert nennen.

In [ ]:
baseline_instruction = (
    "Du bist der PMO-Assistent. Beantworte Fragen nur anhand der Korpusauszüge. "
    "Wenn die Auszüge die Frage nicht beantworten, schreibe: Nicht im Korpus."
)
prepared_dev = prepare_cases(dev_cases)
baseline_dev = run_cases(prepared_dev, baseline_instruction)
show_results(baseline_dev)

Untersuche fehlgeschlagene Checks und lies die Antworten. Vergleiche sie mit den abgerufenen Ausschnitten in `prepared_dev`. Trenne dabei Retrieval-Fehler von Antwortfehlern: Wenn die benötigte Quelle nicht gefunden wurde, kann eine bessere Antwortanweisung die fehlende Evidenz nicht ersetzen.

In [ ]:
for result, item in zip(baseline_dev, prepared_dev):
    if not result["answer_check"] or result["retrieval_hit"] is False:
        print(f"\n--- {result['id']} ---")
        print("Frage:", item["case"]["query"])
        print("Erwartete Belege:", ", ".join(item["case"]["expected_facts"]) or "keine")
        print("Einordnung:", diagnose(result))
        # Kontext als Klartext: er enthaelt Tabellenzeichen, die Markdown umdeuten wuerde
        print("Kontext:\n", item["context"])
        mprint(f"**Antwort:** {result['answer']}")

# 6 | Eine Änderung wählen und vergleichen
---

Die Analyse der Entwicklungsfälle gibt die Richtung vor. Sinnvoll ist eine Anweisungsänderung nur dort, wo die Evidenz im Kontext stand und die Antwort sie unvollständig wiedergibt — ohne Beleg oder ohne Quelle. Nennt die Baseline Aussagen ohne die zugehörige Datei, setzt die Änderung dort an: Sie ergänzt die Pflicht, jede sachliche Aussage einem Dateinamen zuzuordnen. Für Fälle, deren Beleg schon im Kontext fehlt, ist keine Verbesserung zu erwarten. Sie ist ein begründeter Vorschlag, keine garantierte Verbesserung. Verändere möglichst nur einen Aspekt und halte die übrigen Einstellungen konstant.

Der Vergleich meldet zusätzlich `teilregressionen`: eine Stufe, die sich verschlechtert, ohne das Gesamturteil zu kippen — etwa eine Antwort, die ihre Quelle verliert, während sie den Check ohnehin nicht besteht.

In [ ]:
candidate_instruction = (
    baseline_instruction +
    " Nenne zu jeder sachlichen Aussage die Datei, aus der sie stammt."
)
candidate_dev = run_cases(prepared_dev, candidate_instruction)
show_results(candidate_dev)

STUFEN = ("retrieval_hit", "source_cited", "fact_in_context", "fact_check")

def compare_results(baseline: list[dict], candidate: list[dict]) -> dict:
    before = {r["id"]: r for r in baseline}
    after = {r["id"]: r for r in candidate}
    if len(before) != len(baseline) or len(after) != len(candidate) or not before or before.keys() != after.keys():
        raise ValueError("Beide Läufe brauchen dieselben eindeutigen Fall-IDs.")
    return {
        "improvements": [i for i in before if not before[i]["answer_check"] and after[i]["answer_check"]],
        "regressions": [i for i in before if before[i]["answer_check"] and not after[i]["answer_check"]],
        # Verschlechterung einer Teilstufe, die das Gesamturteil nicht umkippt
        "teilregressionen": [f"{i}:{stufe}" for i in before for stufe in STUFEN
                             if before[i][stufe] is True and after[i][stufe] is False],
    }

dev_comparison = compare_results(baseline_dev, candidate_dev)
print("Entwicklungsvergleich:", dev_comparison)

Ein automatischer Treffervergleich ist nur ein erster Hinweis. Beurteile insbesondere, ob die Aussagen von den angeführten Auszügen gedeckt sind und ob Antworten unnötig verweigert werden. Vergleiche wiederholte Läufe, wenn die Modellausgaben schwanken. Wähle die Variante erst nach dieser Prüfung; ändere sie nicht anhand der zurückgehaltenen Fälle.

# 7 | Zurückgehaltene Fälle und Entscheidung
---

Die nächste Zelle führt beide festgelegten Varianten auf denselben zurückgehaltenen Fällen aus. Beurteile anschließend Antworten, Quellen und Regressionen. Die Fallzahl ist für eine belastbare Freigabe zu klein; in der Praxis sind mehr unabhängige Fälle und fachliche Reviews nötig.

In [ ]:
prepared_holdout = prepare_cases(holdout_cases)
baseline_holdout = run_cases(prepared_holdout, baseline_instruction)
candidate_holdout = run_cases(prepared_holdout, candidate_instruction)
print("Baseline:")
show_results(baseline_holdout)
print("Variante:")
show_results(candidate_holdout)
print("Testvergleich:", compare_results(baseline_holdout, candidate_holdout))


**Entscheidung:** Übernimm die Änderung nur, wenn sie nach fachlicher Prüfung hilft und keine wichtigen Fälle verschlechtert. Das Notebook ändert keine produktiven Anweisungen automatisch. Nach der Auswertung sind diese Fälle nicht mehr unabhängig zurückgehalten und müssen für spätere Änderungen durch neue ergänzt werden.

# A | Aufgaben
---



**Grundlagen:** Wähle einen Entwicklungsfall. Vergleiche Frage, Retrieval-Treffer, Antwort und automatischen Check. Erkläre, was der Check zeigt und was er nicht zeigt.

**Erledigt, wenn:** Du einen Fall anhand des tatsächlichen Ausschnitts fachlich beurteilt und die Grenze des Checks benannt hast.

**Aufbau:** Untersuche einen Fall ohne erwartete Korpusantwort. Enthält die Ausgabe neben der Korpusgrenze unbelegte Zusatzbehauptungen? Ergänze einen einfachen Test für einen konkreten beobachteten Fehler und erkläre seine Grenzen.

**Erledigt, wenn:** Dein zusätzlicher Test mit einer passenden und einer fehlerhaften Beispielantwort geprüft wurde.

**Vertiefung:** Entwirf eine alternative Anweisungsänderung anhand der Entwicklungsfälle und vergleiche sie mit der Baseline. Prüfe eine neue, zuvor nicht verwendete Gruppe zurückgehaltener Fälle. Dokumentiere Verbesserungen, Regressionen und die Grenzen deiner Schlussfolgerung.

**Praxis-Transfer-Aufgabe: PMO-Assistent**

Übernimm die RAG-Fälle aus dem Regressionstest-Set von Agent Evaluation und Testing als bekannte Vergleichsbasis. Falls dieses Set nicht vorliegt, verwende die vorhandenen Entwicklungsfälle dieses Notebooks. Übertrage eigene Fälle in das hier verwendete Schema mit `id`, `query`, `expected_source`, `expected_facts` und `out_of_corpus`. Dokumentiere, welche bisherigen Erwartungen durch die Textprüfungen abgedeckt werden und welche weiterhin fachlich geprüft werden müssen.

Wähle anhand der Entwicklungsfälle eine Anweisungsänderung und vergleiche Baseline und Variante zusätzlich auf den bekannten Regressionsfällen. Lege vor dem abschließenden Vergleich eine neue, bisher unbenutzte Gruppe von Prüffällen fest. Bereits zur Entwicklung oder in früheren Auswertungen verwendete Fälle sind keine unabhängigen zurückgehaltenen Fälle. Freigabe- und Sicherheitsfälle aus dem früheren Set bleiben gesondert dokumentiert, soweit der kleine PMO-Graph ihre Prüfung nicht implementiert.

**Ergebnis:** Ein fortgeschriebenes Regressionstest-Set und eine fachlich geprüfte Vergleichstabelle mit Entwicklungsfällen, bekannten Regressionsfällen und neuen zurückgehaltenen Fällen als getrennten Gruppen. Ergänze eine begründete Empfehlung für die fachliche Abnahme durch Projektleiterin Mara Vogt.

**Erledigt, wenn:** Die Fallherkunft und die Prüfgrenzen nachvollziehbar sind, die Anweisung vor der Auswertung der neuen Fälle feststeht und neue Fehler ausgewiesen werden. Ungeprüfte Kontrollpunkte bleiben offen; nach dieser Auswertung sind auch die neuen Fälle für spätere Änderungen nicht mehr unabhängig zurückgehalten.


# B | Dokumente zum Weiterlesen
---



Ergänzende Artikel aus der Kurs-Dokumentation:

- [Loop Engineering](https://ralf-42.github.io/Agenten/04-agenten-implementierung/ablauf-zustand/loop-engineering.html)
- [Evaluation & Observability](https://ralf-42.github.io/Agenten/07-qualitaet-sicherheit/evaluation-observability.html)
- [Prompt Engineering](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/prompt-engineering.html)
- [RAG-Konzepte](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/rag-konzepte.html)